# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key and google_api_key.startswith('sk-proj-') and len(google_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)


There might be a problem with your API key? Please visit the troubleshooting notebook!


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling gemini-3-flash-preview")
    response = gemini.chat.completions.create(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3-flash-preview
Found 10 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'related company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'news/press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/huggingface'},
  {'type': 'github profile', 'url': 'https://github.com/huggingface'},
  {'type': 'organization profile',
   'url': 'https://huggingface.co/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 17 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
3 days ago
•
13.5k
•
1.09k
prism-ml/Ternary-Bonsai-27B-gguf
Updated
1 day ago
•
339k
•
758
prism-ml/Bonsai-27B-gguf
Updated
1 day ago
•
1.26M
•
460
zai-org/GLM-5.2
Updated
17 days ago
•
536k
•
4.14k
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-G

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 14 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling\nUpdated\n3 days ago\n•\n13.5k\n•\n1.09k\nprism-ml/Ternary-Bonsai-27B-gguf\nUpdated\n1 day 

In [17]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 17 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the world’s leading platform for machine learning collaboration. Known as the "Home of Machine Learning," we provide the infrastructure and community tools that allow researchers, developers, and organizations to build, share, and scale AI models, datasets, and applications.

## Our Mission
We are on a mission to democratize good machine learning. We believe that by providing a collaborative ecosystem for the ML community, we can accelerate the pace of innovation and ensure that the future of AI is open and accessible to everyone.

---

## The Platform at a Glance

Hugging Face serves as the central hub for the entire AI lifecycle:

*   **Models:** Browse and download over 2 million open-source models for tasks ranging from natural language processing and computer vision to audio and video generation.
*   **Datasets:** Access more than 500,000 datasets to train, fine-tune, and evaluate your machine learning projects.
*   **Spaces:** Explore over 1 million ML applications. Spaces allow users to host interactive demos of their models directly in the browser.
*   **HuggingChat:** An open-source interface for interacting with the latest large language models.
*   **Infrastructure:** We provide professional tools including Inference Endpoints, Storage Buckets, and specialized hardware solutions to move your projects from research to production faster.

---

## Solutions for Enterprise

Hugging Face is not just for individual researchers; we power the AI strategy of the world's most innovative companies. Our **Team & Enterprise Plans** provide:

*   **Enterprise Support:** Direct access to Hugging Face experts to help navigate the complexities of ML deployment.
*   **Inference Providers:** Scalable, secure environments to run models with high performance.
*   **Private Collaboration:** Securely host and collaborate on internal models and datasets within your organization.
*   **Security & Compliance:** Enterprise-grade tools to manage data and model governance.

---

## Company Culture & Community

At Hugging Face, we are defined by our commitment to the community. Our culture is built on the principles of **openness, collaboration, and rapid innovation.**

*   **Community-Driven:** We host a vibrant ecosystem supported by Discord, community forums, and daily paper discussions to keep our users at the cutting edge.
*   **Educational Focus:** Through our "Learn" initiatives and extensive documentation, we empower the next generation of AI practitioners.
*   **Innovation First:** We are at the forefront of AI trends, supporting the latest breakthroughs in 1-bit LLMs, real-time voice chat, and text-to-video generation.

---

## Join the Future of AI

**For Recruits:**
Hugging Face offers the opportunity to work at the very heart of the AI revolution. You will work alongside the creators of the libraries and tools that the entire industry relies on. If you are passionate about open source and want to shape the future of machine learning, Hugging Face is the place for you.

**For Investors:**
With over 2 million models and a massive, growing user base, Hugging Face is the undisputed leader in the AI platform space. We are the infrastructure layer for the generative AI era, providing the essential tools that every company needs to build their AI future.

---

**Explore the future of AI today at HuggingFace.co**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 12 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the leading platform for the machine learning community, serving as the central hub where the world collaborates on models, datasets, and applications. We are "The Home of Machine Learning," providing the essential infrastructure for the AI revolution.

## The Heart of Open Source AI

Hugging Face is more than just a repository; it is a collaborative ecosystem designed to accelerate the development of artificial intelligence.

*   **Models:** Explore and host over 2 million models, ranging from Large Language Models (LLMs) to specialized computer vision and audio tools.
*   **Datasets:** Access over 500,000 datasets to train, evaluate, and benchmark machine learning projects.
*   **Spaces:** Discover over 1 million AI applications. Spaces allow developers to showcase their ML projects through interactive demos and web applications.
*   **Collaboration:** We offer a central platform for organizations and individuals to host and collaborate on unlimited public and private projects.

## Solutions for Every Scale

Whether you are a solo researcher or a Fortune 500 company, Hugging Face provides the tools to move from development to production.

*   **Hugging Face PRO:** Tailored for individual power users who need extra compute and early access to new features.
*   **Enterprise Hub:** Secure collaboration tools for teams, featuring advanced security, enterprise support, and private model management.
*   **Inference Solutions:** Deploy models instantly with Inference Endpoints or utilize our integrated Inference Providers for scalable production environments.
*   **Storage Buckets:** Specialized storage solutions for the massive amounts of data required for modern AI.

## Our Culture and Community

Hugging Face is built on the belief that AI should be accessible, transparent, and collaborative. Our culture is defined by:

*   **Openness:** We are deeply rooted in the open-source community, hosting daily papers, tutorials, and forums to democratize AI knowledge.
*   **Global Collaboration:** With a massive Discord community and a thriving GitHub presence, we bring together the brightest minds in the industry.
*   **Innovation:** We stay at the cutting edge by integrating the latest hardware and research into our platform, including support for WebGPU kernels and real-time voice technology.

## For Prospective Recruits and Partners

Joining Hugging Face means working at the epicenter of the AI industry. We are a fast-growing team of researchers, engineers, and advocates dedicated to building the tools that will define the next decade of technology.

*   **Prospective Customers:** Accelerate your AI roadmap by leveraging the world’s largest collection of pre-trained models and robust deployment tools.
*   **Investors:** Hugging Face represents the foundational layer of the AI stack, serving as the primary distribution and development point for the entire industry.
*   **Recruits:** If you are passionate about open source, community building, and the future of machine learning, Hugging Face is the place to build it.

**Join the community and help us build the future of AI.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>